# Notebook 3b – Phòng thủ bằng tiền xử lý đầu vào (Input Preprocessing)

**Đề tài 2.21 – Nghiên cứu và phòng thủ trước tấn công đối kháng lên mô hình phân loại ảnh**

Người phụ trách: **Nguyễn Phương Đông** (phòng thủ)

Đặt một phép biến đổi g trước mô hình baseline: `f(g(x))`, với g là

- **Nén JPEG** (chất lượng `config.JPEG_QUALITIES`),
- **Giảm độ sâu bit** – *feature squeezing* (`config.BIT_DEPTHS` bit mỗi kênh),
- **Làm mờ Gaussian** (`config.BLUR_SIGMAS`).

Mỗi cấu hình được đo trong 3 kịch bản:

1. **Ảnh sạch** – phòng thủ làm giảm clean accuracy bao nhiêu?
2. **Kẻ tấn công không biết phòng thủ**: FGSM / PGD ε = 8/255 sinh trên **mô hình gốc** rồi mới đi qua g.
3. **Kẻ tấn công thích nghi (BPDA)**: PGD tấn công trực tiếp `f(g(x))`; với các g không khả vi (JPEG, lượng tử hoá) dùng *Backward Pass Differentiable Approximation* (Athalye et al., 2018): lượt forward dùng g thật, lượt backward coi g ≈ hàm đồng nhất.

> Giao diện dùng chung với phần tấn công: xem `docs/INTERFACE.md`.

## 0. Setup môi trường

- **Colab**: mount Google Drive (checkpoint lưu ở `MyDrive/AdvML_Project/checkpoints` để không mất khi runtime bị ngắt), clone repo vào `/content` nếu chưa có (có rồi thì `git pull`), rồi thêm repo vào `sys.path`.
- **Local**: thêm thư mục gốc repo (thư mục cha của `notebooks/`) vào `sys.path`.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/Eastern2k4/adversarial-attack-defense.git"
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/AdvML_Project", exist_ok=True)

    REPO_DIR = Path("/content/adversarial-attack-defense")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
else:
    cwd = Path.cwd().resolve()
    REPO_DIR = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print("Repo:", REPO_DIR)

### Copy dữ liệu CIFAR-10 từ Drive (nếu có)

Nếu đã để sẵn `cifar-10-python.tar.gz` trong `MyDrive/AdvML_Project/`, copy về `data/` để khỏi tải lại (torchvision sẽ tự giải nén). Không có file thì bỏ qua, dữ liệu sẽ được tải tự động.

In [ ]:
import shutil

DRIVE_DATA = Path("/content/drive/MyDrive/AdvML_Project/cifar-10-python.tar.gz")
LOCAL_DATA = Path("data") / DRIVE_DATA.name
if DRIVE_DATA.exists() and not LOCAL_DATA.exists():
    LOCAL_DATA.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy(DRIVE_DATA, LOCAL_DATA)
    print("Đã copy", DRIVE_DATA, "->", LOCAL_DATA)
else:
    print("Bỏ qua copy dữ liệu (không có file trên Drive hoặc đã có sẵn).")

In [ ]:
import json
from functools import partial

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from src import config
from src.attacks import fgsm, pgd
from src.config import get_device
from src.data import get_eval_subset
from src.defenses import PreprocessedModel, gaussian_blur, jpeg_compress, reduce_bit_depth
from src.models import build_model
from src.utils import set_seed, accuracy_under_attack, load_checkpoint

print("Checkpoint dir:", config.CHECKPOINT_DIR)

## 1. Nạp mô hình baseline và tập đánh giá

In [ ]:
device = get_device()
print("Device:", device)
set_seed(config.SEED)

BASELINE_PATH = config.CHECKPOINT_DIR / config.BASELINE_CKPT
if not BASELINE_PATH.exists():
    raise FileNotFoundError(
        f"Chưa có mô hình baseline tại {BASELINE_PATH}.\n"
        "Hãy chạy notebooks/01_baseline.ipynb để huấn luyện trước, hoặc copy file "
        f"{config.BASELINE_CKPT} vào thư mục {config.CHECKPOINT_DIR} (trên Colab: MyDrive/AdvML_Project/checkpoints)."
    )
model = build_model().to(device)
load_checkpoint(BASELINE_PATH, model, map_location=device)
model.eval()

x_eval, y_eval = get_eval_subset(as_tensors=True)
EPS = max(config.EPSILONS)  # 8/255
EPS_KEY = f"{round(EPS * 255)}/255"
print(f"Eval subset: {len(x_eval)} ảnh | eps = {EPS_KEY}")
print(f"Clean accuracy baseline: {accuracy_under_attack(model, x_eval, y_eval):.2f}%")

## 2. Sinh ảnh đối kháng trên mô hình gốc (kẻ tấn công không biết phòng thủ)

Sinh **một lần** FGSM và PGD-10 trên baseline, sau đó dùng lại cho mọi phép tiền xử lý.

In [ ]:
def generate(model, x, y, attack_fn, batch_size=256):
    """Sinh ảnh đối kháng theo batch, trả về tensor CPU."""
    model.eval()
    out = []
    for i in range(0, len(x), batch_size):
        xb, yb = x[i:i + batch_size].to(device), y[i:i + batch_size].to(device)
        out.append(attack_fn(model, xb, yb).cpu())
    return torch.cat(out)


pgd_attack = partial(pgd, eps=EPS, alpha=config.PGD_ALPHA, steps=config.PGD_STEPS)

set_seed(config.SEED)
x_fgsm = generate(model, x_eval, y_eval, partial(fgsm, eps=EPS))
x_pgd = generate(model, x_eval, y_eval, pgd_attack)
print(f"FGSM trên baseline (không phòng thủ): {accuracy_under_attack(model, x_fgsm, y_eval):.2f}%")
print(f"PGD-{config.PGD_STEPS} trên baseline (không phòng thủ): {accuracy_under_attack(model, x_pgd, y_eval):.2f}%")

### Minh hoạ các phép tiền xử lý trên một ảnh

In [ ]:
transforms = [("none", None, lambda x: x)]
transforms += [("jpeg", q, partial(jpeg_compress, quality=q)) for q in config.JPEG_QUALITIES]
transforms += [("bit_depth", b, partial(reduce_bit_depth, bits=b)) for b in config.BIT_DEPTHS]
transforms += [("blur", s, partial(gaussian_blur, sigma=s)) for s in config.BLUR_SIGMAS]


def label_of(name, param):
    return {"none": "Không xử lý", "jpeg": f"JPEG q={param}",
            "bit_depth": f"{param}-bit", "blur": f"Blur σ={param}"}[name]


img = x_pgd[:1].to(device)
fig, axes = plt.subplots(1, len(transforms), figsize=(2 * len(transforms), 2.4))
for ax, (name, param, g) in zip(axes, transforms):
    ax.imshow(g(img)[0].permute(1, 2, 0).cpu().numpy())
    ax.set_title(label_of(name, param), fontsize=9)
    ax.axis("off")
plt.suptitle(f"Ảnh PGD ({config.CLASSES[int(y_eval[0])]}) sau từng phép tiền xử lý")
plt.tight_layout()
plt.show()

## 3. Đánh giá

Với mỗi cấu hình: clean, FGSM/PGD chuyển từ mô hình gốc, và PGD-10 thích nghi qua `PreprocessedModel(bpda=True)`.

In [ ]:
results = {}
for name, param, g in transforms:
    defended = PreprocessedModel(model, g, bpda=False).eval()
    adaptive = PreprocessedModel(model, g, bpda=True).eval()
    set_seed(config.SEED)
    key = name if param is None else f"{name}_{param}"
    results[key] = {
        "model": f"{config.BASELINE_CKPT.split('.')[0]} + {label_of(name, param)}",
        "transform": name,
        "param": param,
        "clean_acc": accuracy_under_attack(defended, x_eval, y_eval),
        "fgsm": {EPS_KEY: accuracy_under_attack(defended, x_fgsm, y_eval)},
        "pgd": {EPS_KEY: accuracy_under_attack(defended, x_pgd, y_eval)},
        f"pgd_bpda_{EPS_KEY}": accuracy_under_attack(adaptive, x_eval, y_eval, pgd_attack),
    }
    r = results[key]
    print(f"{label_of(name, param):<14s} | clean {r['clean_acc']:6.2f}% | FGSM {r['fgsm'][EPS_KEY]:6.2f}% | "
          f"PGD {r['pgd'][EPS_KEY]:6.2f}% | PGD-BPDA {r[f'pgd_bpda_{EPS_KEY}']:6.2f}%")

In [ ]:
df = pd.DataFrame([
    {
        "Phép biến đổi": label_of(r["transform"], r["param"]),
        "Sạch": r["clean_acc"],
        "FGSM (không thích nghi)": r["fgsm"][EPS_KEY],
        "PGD (không thích nghi)": r["pgd"][EPS_KEY],
        "PGD-BPDA (thích nghi)": r[f"pgd_bpda_{EPS_KEY}"],
    }
    for r in results.values()
]).set_index("Phép biến đổi")
df.round(2)

In [ ]:
ax = df.plot(kind="bar", figsize=(13, 5), width=0.8)
ax.set_ylabel("Accuracy (%)"); ax.set_xlabel(""); ax.set_ylim(0, 100)
ax.set_title(f"Tiền xử lý đầu vào trên baseline – ε = {EPS_KEY}")
ax.grid(axis="y", alpha=.3); ax.legend(loc="upper right")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
config.RESULTS_DIR.mkdir(parents=True, exist_ok=True)
plt.savefig(config.RESULTS_DIR / "preprocessing_accuracy.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Lưu kết quả

`results/preprocessing_metrics.json`: mỗi cấu hình một mục (khóa `none`, `jpeg_90`, `bit_depth_4`, `blur_0.5`, ...) theo cấu trúc chung (`model`, `clean_acc`, `fgsm`, `pgd`) cộng thêm `transform`, `param`, `pgd_bpda_8/255`.

In [ ]:
with open(config.RESULTS_DIR / "preprocessing_metrics.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print("Đã lưu", config.RESULTS_DIR / "preprocessing_metrics.json")

In [ ]:
# Trên Colab: sao lưu thư mục results/ lên Drive để không mất khi runtime kết thúc.
if IN_COLAB:
    dst = Path("/content/drive/MyDrive/AdvML_Project/results")
    shutil.copytree(config.RESULTS_DIR, dst, dirs_exist_ok=True)
    print("Đã sao lưu results/ lên", dst)

## 5. Nhận xét (điền sau khi có số liệu)

Gợi ý các điểm cần phân tích trong báo cáo:

- **Đánh đổi clean accuracy**: phép biến đổi càng mạnh (JPEG q thấp, ít bit, σ lớn) thì clean accuracy càng giảm.
- **Kẻ tấn công không biết phòng thủ**: nhiễu đối kháng tần số cao thường bị JPEG/blur/lượng tử hoá "xoá" một phần → accuracy dưới FGSM/PGD tăng so với không xử lý.
- **Kẻ tấn công thích nghi (BPDA)**: nếu accuracy rơi trở lại gần mức không phòng thủ, thì tiền xử lý chỉ tạo cảm giác an toàn giả (*gradient obfuscation*, Athalye et al. 2018) – khác với adversarial training (Notebook 3a) vốn bền vững cả khi kẻ tấn công biết rõ mô hình.